[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/3-context-engineering.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 3. Context engineering

The prompt can hold titles. Bodies load when a question needs them.

`list_help_articles` returns ids, titles, and sizes. `read_help_article` returns one body, and it refuses `preserving-mega-guide` because that file is over the cap. The model passes an article id, not a path.

A customer asks what the short help article says about an opened jar. The catalog tool and the topic lookup from tutorial 2 are imported above this lesson's tools.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## Titles, then one body

The mega-guide is refused before the model runs. The short article loads.

In [ ]:
import tutorial.common.facts
import tutorial.common.catalog
import tutorial.common.articles  # registers list_help_articles and read_help_article

from tutorial.common.articles import ARTICLE_CAP, catalog_chars, list_help_articles, read_help_article
from tutorial.common.get_db import reset_db
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent

print(list_help_articles({}))
print("cap", ARTICLE_CAP, "paste-all", catalog_chars())
refused = read_help_article({"name": "preserving-mega-guide"})
print(refused)
opened = read_help_article({"name": "opened-jars"})
print(opened)
check(catalog_chars() > ARTICLE_CAP, "pasting every article is larger than the cap")
check(refused.startswith("ERROR:"), "the mega-guide is refused")
check(opened.startswith("ARTICLE:"), "the short article loads")

## Run

The customer asks for the short article. The system prompt already lists titles, not bodies.

In [ ]:
reset_db()
result = run_agent(
    "I opened a jar. What does your short help article say about returns?",
    system_text(),
)
print("ANSWER:", result["text"])
log = "\n".join(result["tool_log"])
check("read_help_article" in log, "a body was loaded on purpose")
check("preserving-mega-guide" not in log, "the mega-guide was not the article that was read")
check("final sale" in result["text"].lower(), "the answer uses the short article")